## Configuración para poder importar desde el src/*

In [1]:
import os
import sys
from pathlib import Path
ROOT = Path().resolve()
while ROOT.name != "pdf-key-extraction":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

In [2]:
from dataclasses import dataclass
import json
from PIL import Image
from common.common_types import LayoutElement
from common.data_storage import DataStorage

@dataclass
class PageSample:
    images: list[Image.Image]
    elements: list[LayoutElement]

paths = DataStorage.find_json_paths()
dataset: list[PageSample] = []
for path in paths:
    with open(path) as f:
        data = json.load(f)
        images = DataStorage.get_images(path.stem)
        dataset.append(PageSample(images=images, elements=data))


In [3]:
all_labels = set()
for doc in dataset:
    for e in doc.elements:
        all_labels.add(e["label"])

label_list = sorted(list(all_labels))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

print(f"Total facturas: {len(dataset)}")
print(f"Etiquetas: {label_list}")

Total facturas: 77
Etiquetas: ['FIELD_KEY_ADDRESS', 'FIELD_KEY_AMOUNT', 'FIELD_KEY_DATE', 'FIELD_KEY_EMAIL', 'FIELD_KEY_ID', 'FIELD_KEY_NAME', 'FIELD_KEY_TEXT', 'FIELD_VALUE_ADDRESS', 'FIELD_VALUE_AMOUNT', 'FIELD_VALUE_DATE', 'FIELD_VALUE_EMAIL', 'FIELD_VALUE_ID', 'FIELD_VALUE_NAME', 'FIELD_VALUE_TEXT', 'HEADER_PRODUCT_CODE', 'HEADER_PRODUCT_CODE_AUX', 'HEADER_PRODUCT_DETAIL', 'HEADER_PRODUCT_DISCOUNT', 'HEADER_PRODUCT_NAME', 'HEADER_PRODUCT_PRICE', 'HEADER_PRODUCT_QUANTITY', 'HEADER_PRODUCT_SUBSIDY', 'HEADER_PRODUCT_TOTAL', 'HEADER_PRODUCT_WITHOUT_SUBSIDY', 'ITEM_PRODUCT_CODE', 'ITEM_PRODUCT_CODE_AUX', 'ITEM_PRODUCT_DETAIL', 'ITEM_PRODUCT_DISCOUNT', 'ITEM_PRODUCT_NAME', 'ITEM_PRODUCT_PRICE', 'ITEM_PRODUCT_QUANTITY', 'ITEM_PRODUCT_SUBSIDY', 'ITEM_PRODUCT_TOTAL', 'ITEM_PRODUCT_WITHOUT_SUBSIDY', 'O']


## Preparación

In [4]:
from transformers import LayoutLMv3Processor

processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

c:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
from PIL import Image

def prepare_document(elements):
    words = [e["text"] for e in elements]
    boxes = [e["normalized_bbox"] for e in elements]
    labels = [label2id[e["label"]] for e in elements]
    return words, boxes, labels



def encode_document(image:Image.Image,elements: list[LayoutElement]):
    words, boxes, labels = prepare_document(elements)
  
    encoding = processor(
        images=image,
        text=words,
        boxes=boxes,
        word_labels=labels,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )
    return encoding




## Entrenamiento

In [6]:
from transformers import LayoutLMv3ForTokenClassification, TrainingArguments, Trainer
import torch

model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

c:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
from torch.utils.data import Dataset as TorchDataset

def extract_per_page(page:PageSample):
    separated = []
    for index,image in enumerate(page.images):
        current_page = index + 1 
        current_elements = [e for e in page.elements if e["page"] == current_page]
        separated.append((image, current_elements))
    return separated

class InvoiceDataset(TorchDataset):
    def __init__(self, documents):
        self.documents = documents

    def __getitem__(self, idx):
        image, elements = self.documents[idx]
        encoding = encode_document(image, elements)
        return {k: v.squeeze(0) for k, v in encoding.items()}

    def __len__(self):
        return len(self.documents)
    
split = int(len(dataset) * 0.8)

train_data = dataset[:split]
eval_data = dataset[split:]

train_data_final = []

for page in train_data:
    train_data_final.extend(extract_per_page(page))

eval_data_final = []
for page in eval_data:
    eval_data_final.extend(extract_per_page(page))


train_dataset = InvoiceDataset(train_data_final)
val_dataset = InvoiceDataset(eval_data_final)

print()
print(f"Train: {len(train_data)} | Val: {len(eval_data)}")
print(f"Train pages: {len(train_data_final)} | Val pages: {len(eval_data_final)}")


Train: 61 | Val: 16
Train pages: 94 | Val pages: 20


In [8]:




from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import LayoutLMv3Processor


def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics(p):
    predictions, labels = p

    true_predictions = [
        id2label[pred_id]
        for pred, lab in zip(predictions, labels)
        for (pred_id, l) in zip(pred, lab) if l != -100
    ]
    true_labels = [
        id2label[l]
        for pred, lab in zip(predictions, labels)
        for (pred_id, l) in zip(pred, lab) if l != -100
    ]

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_predictions, average="weighted", zero_division=0
    )
    acc = accuracy_score(true_labels, true_predictions)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": acc,
    }
training_args = TrainingArguments(
    output_dir="./model-output",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    save_steps=50,
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    eval_accumulation_steps=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
   
)

trainer.train()

trainer.save_model("./model-output/final")
processor.save_pretrained("./model-output/final")

  0%|          | 0/470 [00:00<?, ?it/s]c:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\transformers\modeling_utils.py:1051: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
  2%|▏         | 10/470 [00:07<04:37,  1.66it/s]

{'loss': 3.1063, 'grad_norm': 4.586323261260986, 'learning_rate': 4.893617021276596e-05, 'epoch': 0.21}


  4%|▍         | 20/470 [00:12<04:17,  1.75it/s]

{'loss': 2.1073, 'grad_norm': 5.2453179359436035, 'learning_rate': 4.787234042553192e-05, 'epoch': 0.43}


  6%|▋         | 30/470 [00:18<04:12,  1.74it/s]

...

                                                 
100%|██████████| 470/470 [04:57<00:00,  1.81it/s]

{'eval_loss': 0.02390996739268303, 'eval_precision': 0.997520206569387, 'eval_recall': 0.9974603174603175, 'eval_f1': 0.9974635589613557, 'eval_accuracy': 0.9974603174603175, 'eval_runtime': 1.3719, 'eval_samples_per_second': 14.578, 'eval_steps_per_second': 7.289, 'epoch': 10.0}


100%|██████████| 470/470 [04:58<00:00,  1.57it/s]


{'train_runtime': 298.492, 'train_samples_per_second': 3.149, 'train_steps_per_second': 1.575, 'train_loss': 0.22747180588701937, 'epoch': 10.0}


[]